# GeoDiff-GAN on Kaggle

This notebook clones the project, prepares Sentinel-2 L2A RGB patches, optionally captions them with Qwen3-VL-8B, trains the staged model, and evaluates uncertainty.

**Before running:** enable Internet, select a GPU accelerator, and attach a Kaggle dataset containing extracted Sentinel-2 L2A `.SAFE` directories. The quick run validates integration only. Set `FAST_DEV_RUN = False` for the full architecture and paper-scale epoch defaults.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

def run(command, cwd=None):
    """Run a command with visible, copyable output."""
    command = list(map(str, command))
    project_modules = {
        "geodiff-prepare": "geodiff_gan.cli.prepare_sentinel",
        "geodiff-caption": "geodiff_gan.cli.caption_qwen",
        "geodiff-train": "geodiff_gan.cli.train",
        "geodiff-infer": "geodiff_gan.cli.infer",
        "geodiff-evaluate": "geodiff_gan.cli.evaluate",
        "geodiff-baselines": "geodiff_gan.cli.baselines",
        "geodiff-debug": "geodiff_gan.cli.debug",
    }
    if command and command[0] in project_modules and shutil.which(command[0]) is None:
        command = [sys.executable, "-m", project_modules[command[0]], *command[1:]]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

REPOSITORY_URL = "https://github.com/OWNER/geodiff-gan-sentinel2.git"
REPOSITORY_DIR = Path("/kaggle/working/geodiff-gan-sentinel2")
SENTINEL_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/geodiff-gan-output")

FAST_DEV_RUN = True
RUN_DATA_PREPARATION = True
RUN_CAPTIONING = False
CAPTION_LIMIT_PER_SPLIT = 200 if FAST_DEV_RUN else None
STAGES_TO_RUN = ["base", "vae", "diffusion", "joint"]
# Add "edit" only after generating meaningful captions.

PATCH_SIZE = 512
PATCH_STRIDE = 384
MINIMUM_VALID_FRACTION = 0.95
WORK_ROOT.mkdir(parents=True, exist_ok=True)


## 1. Clone and install

The package is installed with `--no-deps` so Kaggle's CUDA-enabled PyTorch is not replaced.

In [ ]:
import os

if not REPOSITORY_DIR.exists():
    run(["git", "clone", "--depth", "1", REPOSITORY_URL, REPOSITORY_DIR])
else:
    print(f"Using existing clone: {REPOSITORY_DIR}")

run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], cwd=REPOSITORY_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "transformers>=4.57.0"])
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPOSITORY_DIR)
os.chdir(REPOSITORY_DIR)
print("Repository:", Path.cwd())


In [ ]:
import torch

GPU_COUNT = torch.cuda.device_count()
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", GPU_COUNT)
for index in range(GPU_COUNT):
    properties = torch.cuda.get_device_properties(index)
    print(index, properties.name, f"{properties.total_memory / 2**30:.1f} GB")
if GPU_COUNT == 0:
    raise RuntimeError("Enable a Kaggle GPU accelerator before training.")


## 2. Prepare Sentinel-2 patches

The command searches recursively below `/kaggle/input`, reads `.SAFE` products by window, applies the Scene Classification Layer mask, and writes patches to `/kaggle/working`. Complete MGRS tiles are assigned to only one split.

In [ ]:
import importlib.util
import json
from collections import Counter

safe_products = sorted(SENTINEL_INPUT.rglob("*.SAFE"))
print(f"Found {len(safe_products)} SAFE products")
for product in safe_products[:10]:
    print(" -", product)

PATCH_ROOT = WORK_ROOT / "patches"
MANIFEST = WORK_ROOT / "manifest.jsonl"
if RUN_DATA_PREPARATION and not MANIFEST.exists():
    if not safe_products:
        raise FileNotFoundError(
            "No extracted .SAFE directory was found below /kaggle/input. "
            "Attach a dataset containing Sentinel-2 L2A SAFE directories."
        )
    if importlib.util.find_spec("geodiff_gan") is None:
        raise RuntimeError(
            "GeoDiff-GAN is not installed in this Kaggle session. "
            "Run the '1. Clone and install' cell, then rerun this cell."
        )
    run([
        sys.executable, "-m", "geodiff_gan.cli.prepare_sentinel",
        "--input", SENTINEL_INPUT,
        "--output", PATCH_ROOT,
        "--manifest", MANIFEST,
        "--patch-size", PATCH_SIZE,
        "--stride", PATCH_STRIDE,
        "--minimum-valid-fraction", MINIMUM_VALID_FRACTION,
    ])

if not MANIFEST.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST}")
records = [json.loads(line) for line in MANIFEST.read_text().splitlines() if line.strip()]
if not records:
    raise RuntimeError(
        "Data preparation produced no valid patches. Inspect the SAFE product, SCL masks, "
        "and MINIMUM_VALID_FRACTION before training."
    )
split_counts = Counter(record["split"] for record in records)
tile_counts = Counter(record["tile_id"] for record in records)
print("Patches:", len(records))
print("Splits:", split_counts)
print("Tiles:", tile_counts)
missing_splits = [split for split in ("train", "val", "test") if not split_counts[split]]
if missing_splits:
    message = (
        f"Missing tile-level splits: {missing_splits}. Found {len(tile_counts)} unique MGRS "
        "tile(s). A one-tile dataset is suitable only for FAST_DEV_RUN pipeline debugging; "
        "it cannot measure geographic generalization."
    )
    if not FAST_DEV_RUN:
        raise RuntimeError(message + " Attach more geographically separated SAFE products.")
    print("WARNING:", message)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_record = records[0]
with np.load(sample_record["patch"]) as sample_data:
    sample_hr = sample_data["hr"]
if sample_hr.shape[0] == 3:
    sample_hr = sample_hr.transpose(1, 2, 0)
plt.figure(figsize=(6, 6))
plt.imshow(np.clip(sample_hr, 0, 1))
plt.title(f'{sample_record["tile_id"]} | {sample_record["split"]}')
plt.axis("off");


## 3. Optional Qwen3-VL-8B captions

Captioning runs as a subprocess, so GPU memory is released before training. It uses the official `Qwen/Qwen3-VL-8B-Instruct` checkpoint with 4-bit bitsandbytes quantization. Set `RUN_CAPTIONING = True` above. Internet must be enabled.

In [ ]:
CAPTIONS = WORK_ROOT / "captions.jsonl"
if RUN_CAPTIONING:
    for split in ("train", "val", "test"):
        command = [
            sys.executable, "-m", "geodiff_gan.cli.caption_qwen",
            "--manifest", MANIFEST,
            "--output", CAPTIONS,
            "--model", "Qwen/Qwen3-VL-8B-Instruct",
            "--split", split,
        ]
        if CAPTION_LIMIT_PER_SPLIT is not None:
            command.extend(["--limit", CAPTION_LIMIT_PER_SPLIT])
        run(command)
else:
    print("Captioning skipped. Prompt conditioning will use null prompts.")
print("Caption file:", CAPTIONS if CAPTIONS.exists() else None)


## 4. Build Kaggle runtime configs

Fast mode uses the compact hash-text smoke architecture and one epoch per stage. Full mode uses the 81.8M-parameter architecture, frozen SigLIP, and the paper-scale epoch schedule.

In [ ]:
import copy
import yaml

with (REPOSITORY_DIR / "configs/default.yaml").open() as handle:
    runtime_config = yaml.safe_load(handle)

runtime_config["data"]["manifest"] = str(MANIFEST)
runtime_config["data"]["captions"] = str(CAPTIONS) if CAPTIONS.exists() else None
runtime_config["training"]["batch_size"] = 1
runtime_config["training"]["gradient_accumulation"] = max(1, 16 // GPU_COUNT)
runtime_config["training"]["num_workers"] = 2
runtime_config["training"]["amp"] = True
runtime_config["training"]["gradient_checkpointing"] = True

if FAST_DEV_RUN:
    with (REPOSITORY_DIR / "configs/smoke.yaml").open() as handle:
        smoke = yaml.safe_load(handle)
    runtime_config["model"].update(smoke["model"])
    runtime_config["text_encoder"] = smoke["text_encoder"]
    runtime_config["training"].update(smoke["training"])
    runtime_config["training"]["batch_size"] = 1
    runtime_config["training"]["gradient_accumulation"] = 1

EPOCHS = (
    {"base": 1, "vae": 1, "diffusion": 1, "joint": 1, "edit": 1}
    if FAST_DEV_RUN
    else {"base": 20, "vae": 20, "diffusion": 80, "joint": 20, "edit": 10}
)
CONFIG_ROOT = WORK_ROOT / "configs"
RUN_ROOT = WORK_ROOT / "runs"
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(yaml.safe_dump(runtime_config, sort_keys=False)[:3000])


## 5. Train selected stages

Each stage initializes from the previous stage checkpoint. On a dual-GPU Kaggle runtime, the notebook uses `torchrun`; otherwise it uses one GPU. The edit stage should only be enabled after caption generation.

In [ ]:
def train_stage(stage, initial_checkpoint=None):
    config = copy.deepcopy(runtime_config)
    output_dir = RUN_ROOT / stage
    config["training"]["stage"] = stage
    config["training"]["epochs"] = EPOCHS[stage]
    config["training"]["output_dir"] = str(output_dir)
    config["training"]["init_checkpoint"] = (
        str(initial_checkpoint) if initial_checkpoint is not None else None
    )
    config["training"]["resume"] = None
    if stage == "edit":
        config["training"]["learning_rate"] = 1e-5
        config["training"]["loss_weights"]["consistency"] = 0.2
    elif stage == "joint":
        config["training"]["learning_rate"] = 2e-5
    config_path = CONFIG_ROOT / f"{stage}.yaml"
    with config_path.open("w") as handle:
        yaml.safe_dump(config, handle, sort_keys=False)

    module_command = ["-m", "geodiff_gan.cli.train", "--config", config_path]
    if GPU_COUNT > 1:
        command = ["torchrun", "--standalone", f"--nproc_per_node={GPU_COUNT}", *module_command]
    else:
        command = [sys.executable, *module_command]
    run(command, cwd=REPOSITORY_DIR)
    checkpoints = sorted(output_dir.glob(f"{stage}_epoch_*.pt"))
    if not checkpoints:
        raise RuntimeError(f"No checkpoint produced for stage {stage}")
    return checkpoints[-1]

if "edit" in STAGES_TO_RUN and not CAPTIONS.exists():
    raise RuntimeError("Generate Qwen3-VL captions before training edit mode.")

CHECKPOINTS = {}
previous_checkpoint = None
for stage in STAGES_TO_RUN:
    previous_checkpoint = train_stage(stage, previous_checkpoint)
    CHECKPOINTS[stage] = previous_checkpoint
    print(stage, "->", previous_checkpoint)
CHECKPOINTS


## 6. Evaluate stochastic SR and baselines

In [ ]:
split_counts = Counter(record["split"] for record in records)
evaluation_split = "test" if split_counts["test"] else ("val" if split_counts["val"] else "train")
evaluation_checkpoint = CHECKPOINTS.get("joint", previous_checkpoint)
evaluation_config = CONFIG_ROOT / f"{STAGES_TO_RUN[-1]}.yaml"
EVAL_ROOT = WORK_ROOT / "evaluation"

run([
    sys.executable, "-m", "geodiff_gan.cli.evaluate",
    "--config", evaluation_config,
    "--checkpoint", evaluation_checkpoint,
    "--output", EVAL_ROOT,
    "--split", evaluation_split,
    "--samples", 2 if FAST_DEV_RUN else 8,
    "--steps", 2 if FAST_DEV_RUN else 20,
    "--mode", "sr",
    "--limit", 4 if FAST_DEV_RUN else 100,
])

base_checkpoint = CHECKPOINTS.get("base")
run([
    sys.executable, "-m", "geodiff_gan.cli.baselines",
    "--config", evaluation_config,
    "--base-checkpoint", base_checkpoint,
    "--output", WORK_ROOT / "baselines.json",
    "--split", evaluation_split,
    "--limit", 4 if FAST_DEV_RUN else 100,
])
print((EVAL_ROOT / "metrics.json").read_text())
print((WORK_ROOT / "baselines.json").read_text())


## 7. Generate visual diagnostics

This exports exact tensor statistics, NaN/Inf checks, the evidence gate, LR feature maps, selected diffusion states, residual strength, and LR consistency before and after back-projection. Share this complete folder when requesting architecture feedback.

In [ ]:
from IPython.display import Image as DisplayImage, display

DEBUG_ROOT = WORK_ROOT / "debug_patch_0"
run([
    sys.executable, "-m", "geodiff_gan.cli.debug",
    "--config", evaluation_config,
    "--checkpoint", evaluation_checkpoint,
    "--output", DEBUG_ROOT,
    "--split", evaluation_split,
    "--index", 0,
    "--mode", "sr",
    "--steps", 2 if FAST_DEV_RUN else 20,
    "--diffusion-every", 1 if FAST_DEV_RUN else 5,
])

for diagnostic_image in ("overview.png", "features.png", "diffusion_trajectory.png"):
    path = DEBUG_ROOT / diagnostic_image
    if path.exists():
        display(DisplayImage(filename=str(path)))
print((DEBUG_ROOT / "summary.txt").read_text())


## 8. Export artifacts

The archive contains runtime configs, checkpoints, captions, metrics, uncertainty maps, and the visual diagnostic report. Large extracted patch files are excluded.

In [ ]:
import shutil

EXPORT_ROOT = Path("/kaggle/working/geodiff-gan-results")
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
EXPORT_ROOT.mkdir(parents=True)
for name in ("configs", "runs", "evaluation", "debug_patch_0"):
    source = WORK_ROOT / name
    if source.exists():
        shutil.copytree(source, EXPORT_ROOT / name)
for file_name in ("manifest.jsonl", "captions.jsonl", "baselines.json"):
    source = WORK_ROOT / file_name
    if source.exists():
        shutil.copy2(source, EXPORT_ROOT / file_name)
archive = shutil.make_archive("/kaggle/working/geodiff-gan-results", "zip", EXPORT_ROOT)
print("Exported:", archive)
